# 01 - Policy Renewals 30d (v2, generated 2026-10-13)

Fabric folder: Dev_CGP > Notebooks > 03 - EADM > Collaboration. Run order: this notebook first, then 02 - Friction Frame. v1 was "SEG - Policy
Renewals 30d"; v2 is the same logic (SaleCount >= 1, cancellation-on-due rule, chains through rebooks,
one Python call per partition) under the new name.

A ±30-day equivalent of `edw.tbl_fact_policy_renewals`: **one row per policy term**
(policy code x renewal date), from `FirstDueDate` onwards, with the SCV customer key, the product, the current
policy, the new policy and the dates. Output: `stg.seg_policy_renewals_30d`. Writes `stg` only.

**Renewal rule** (agreed for segmentation V3, 2026-10-12): a policy that lapsed, or was cancelled within 30 days of
its renewal date, is **retained** when the same customer bought a policy of the same product starting within
**±30 days** of the end date. The new policy is the same code renewed, or the newly bought one. Matching is
one-to-one per customer and product, best evidence first:

| tier | evidence |
|---|---|
| 1 | the same policy code renewed |
| 2 | EDW lineage: the new policy's sales row names the old code, or EDW's TYPolicyCode |
| 3 | same vehicle registration (Motor / Van) or same address key (Home) |
| 4 | date only, nearest first |

A renewal sale naming a different predecessor only matches that predecessor. A new code that is Temporary or NB Not
Taken Up never counts; the same code renewing always counts.

**Policy terms** come from the month-end stock (`edw.tbl_fact_policy_mvt`, `CloseNum = 1`, every month ID). Travel terms come from
`ods.policy`, as in VV. Policies cancelled more than 30 days before renewal are kept with `due_flag = 0`.
Rows whose end date is less than 30 days before the latest sale in the data are `outcome_status = 'pending'`.

**Renewal and chains** (agreed 2026-10-13): a term is renewed when the next term's renewal date is within ±30 days of
its renewal due date + 1 year, or of its **cancellation date** + 1 year when it was cancelled within 30 days of
renewal. All terms are 12 months. A term **cancelled mid-term** and replaced within ±30 days is **rebooked**: kept
separate, not a renewal, but the chain runs through it. `root_policy_code` / `root_due_date` / `chain_seq` follow
the policy from its root through every renewal and rebook, including changes of policy code; `previous_policy_code`
points back.
EDW's `PolicyOfferNum` / `PolicyRetNum`, its TY policy code and its offered prices are carried for comparison.

Parameters

In [ ]:
Set FirstDueDate = '2015-03-01' ;
Set MatchDays = 30 ;

Keys. Client code -\> SCV key from ods.scveadmlinkfinal (Relay, SourceSystemId = 1: one key per client code); policy reference -\> SCV key as the fallback (Travel customers are not in Relay); client code -\> address key.

In [ ]:
CREATE OR REPLACE TABLE stg.pr30_scv_client AS
SELECT TRIM(ChillSourceCustomerReference) AS ClientCode, MIN(SCV_Customer_Key) AS SCV_Customer_Key
FROM   ods.scveadmlinkfinal
WHERE  SourceSystemId = 1 AND ChillSourceCustomerReference IS NOT NULL
GROUP  BY TRIM(ChillSourceCustomerReference)
;

CREATE OR REPLACE TABLE stg.pr30_scv_link AS
SELECT TRIM(ChillSourcePolicyReference) AS PolicyCode, MIN(SCV_Customer_Key) AS SCV_Customer_Key
FROM   ods.scveadmlinkfinal
WHERE  ChillSourcePolicyReference IS NOT NULL
GROUP  BY TRIM(ChillSourcePolicyReference)
;

CREATE OR REPLACE TABLE stg.pr30_addr AS
SELECT TRIM(SourceSystemReference) AS ClientCode, MIN(ChillSourceAddressKey) AS addr_key
FROM   ods.scv_customer_key
WHERE  SourceSystemId = 1 AND SourceSystemReference IS NOT NULL
GROUP  BY TRIM(SourceSystemReference)
;

CREATE OR REPLACE TABLE stg.pr30_policy AS
SELECT TRIM(PolicyCode) AS PolicyCode, MAX(TRIM(ClientCode)) AS ClientCode, MAX(PolicyTypeGroup) AS product,
       MAX(PolicyStatusDesc) AS PolicyStatusDesc, MAX(PolicyCancelDate) AS PolicyCancelDate
FROM   ods.policy
GROUP  BY TRIM(PolicyCode)
;

Policy terms (all products except Travel): every (policy code, renewal date) in the month-end stock (mvt, CloseNum = 1). EffectiveDate is carried as month ID (YYYYMM). About 6k codes have their renewal date moved within a term (two dates under 300 days apart): the date seen in the latest month ID is kept.

In [ ]:
CREATE OR REPLACE TABLE stg.pr30_terms AS
WITH t AS (
    SELECT TRIM(PolicyCode) AS policy_code, PolicyRenewDateAdj AS due_date,
           MAX(TRIM(ClientCode)) AS ClientCode, MAX(NULLIF(TRIM(VehicleRegNumber), '')) AS reg,
           MIN(CAST(date_format(EffectiveDate, 'yyyyMM') AS INT)) AS first_month_id,
           MAX(CAST(date_format(EffectiveDate, 'yyyyMM') AS INT)) AS last_month_id
    FROM   edw.tbl_fact_policy_mvt
    WHERE  CloseNum = 1 AND PolicyCode IS NOT NULL AND PolicyRenewDateAdj IS NOT NULL
    GROUP  BY TRIM(PolicyCode), PolicyRenewDateAdj
),
n AS (
    SELECT t.*,
           LAG(due_date)       OVER (PARTITION BY policy_code ORDER BY due_date) AS prev_due,
           LAG(last_month_id)  OVER (PARTITION BY policy_code ORDER BY due_date) AS prev_last,
           LEAD(due_date)      OVER (PARTITION BY policy_code ORDER BY due_date) AS next_due,
           LEAD(last_month_id) OVER (PARTITION BY policy_code ORDER BY due_date) AS next_last
    FROM   t
)
SELECT policy_code, due_date, ClientCode, reg, first_month_id, last_month_id
FROM   n
WHERE  due_date >= ${FirstDueDate}
AND    NOT (next_due IS NOT NULL AND datediff(next_due, due_date) < 300 AND next_last >= last_month_id)
AND    NOT (prev_due IS NOT NULL AND datediff(due_date, prev_due) < 300 AND prev_last >  last_month_id)
;

Due rows (all products except Travel). The cancellation date in ods.policy (Relay pl_cancel_date, the \*effective\* date) belongs to the code's latest term, so it is used only when it falls inside this term: after the term start and before the renewal date. A cancellation effective on or after the renewal date is Relay closing the term at the end of cover (over half of those terms renewed, often to a new code): the term reached its renewal date (agreed 2026-10-13).

**Notes**

- `COALESCE(d.cancel_date, d.due_date) AS end_date,`: cancelled: anchor on the cancellation

In [ ]:
CREATE OR REPLACE TABLE stg.pr30_due AS
WITH d AS (
    SELECT t.*, p.product,
           CASE WHEN CAST(p.PolicyCancelDate AS DATE) >  add_months(t.due_date, -12)
                 AND CAST(p.PolicyCancelDate AS DATE) <  t.due_date
                THEN CAST(p.PolicyCancelDate AS DATE) END AS cancel_date
    FROM   stg.pr30_terms t
    JOIN   stg.pr30_policy p ON p.PolicyCode = t.policy_code
    WHERE  p.product <> 'Travel'
)
SELECT d.*,
       add_months(d.due_date, -12)                                                         AS term_start,
       COALESCE(d.cancel_date, d.due_date)                                                 AS end_date,
       CASE WHEN d.cancel_date IS NULL                                  THEN 'reached renewal date'
            WHEN d.cancel_date >= date_sub(d.due_date, ${MatchDays})    THEN 'cancelled near renewal'
            ELSE 'cancelled mid-term' END                                                  AS end_type,
       CASE WHEN d.cancel_date < date_sub(d.due_date, ${MatchDays}) THEN 0 ELSE 1 END         AS due_flag,
       c.SCV_Customer_Key, a.addr_key,
       r.PolicyOfferNum AS edw_offered, r.PolicyRetNum AS edw_retained, NULLIF(TRIM(r.TYPolicyCode), '') AS edw_ty_policy_code,
       r.RenEURPremOffer AS edw_offered_premium, r.RenEURFee AS edw_offered_fee
FROM   d
LEFT JOIN stg.pr30_scv_client c ON c.ClientCode = d.ClientCode
LEFT JOIN stg.pr30_addr       a ON a.ClientCode = d.ClientCode
LEFT JOIN edw.tbl_fact_policy_renewals_rebuilt r
       ON TRIM(r.PolicyCode) = d.policy_code AND r.LYPolicyRenewDateAdj = d.due_date
;

New policies (all products except Travel): one row per active sale. term_start = renewal date minus 12 months.

**Notes**

- Real sales only: SaleCount \>= 1. EDW (usp_Load_fact_Policy_Trans, MI1210) sets ReportingSaleCategory per month from the month-end close flag and the month's net sale count, so 'Cat C: Cancelled Sale' holds both sales cancelled in the month they were sold (SaleCount 1, kept) and cancellation records (SaleCount 0, e.g. JOHJ7A001's January 2024 row); corrections (Cat B / D) are 0 or -1.

In [ ]:
CREATE OR REPLACE TABLE stg.pr30_sale_cand AS
SELECT TRIM(s.PolicyCode)                    AS cand_code,
       p.PolicyStatusDesc                    AS cand_status,
       s.ReportingSaleType, s.ReportingSaleDate,
       NULLIF(TRIM(s.LYPolicyCode), '')      AS ly_code,
       NULLIF(TRIM(s.VehicleRegNumber), '')  AS reg,
       add_months(s.PolicyRenewDateAdj, -12) AS term_start,
       s.PolicyRenewDateAdj                  AS cand_due_date,
       p.product, c.SCV_Customer_Key, a.addr_key,
       s.EURGrossPremium, s.EURCommission, s.EURFees
FROM   edw.tbl_fact_policy_sales s
JOIN   stg.pr30_policy p ON p.PolicyCode = TRIM(s.PolicyCode)
LEFT JOIN stg.pr30_scv_client c ON c.ClientCode = p.ClientCode
LEFT JOIN stg.pr30_addr       a ON a.ClientCode = p.ClientCode
WHERE  p.product <> 'Travel' AND s.PolicyRenewDateAdj IS NOT NULL
AND    COALESCE(s.SaleCount, 0) >= 1
;

Travel terms, due rows and new policies, from ods.policy as in VV (each Travel term is its own policy code). Void flag from ods.policytravel, latest row per insurer policy number (VV cell 2).

In [ ]:
CREATE OR REPLACE TABLE stg.pr30_travel_void AS
SELECT InsPolNumber, void_flag FROM (
  SELECT TRIM(InsPolicyCode) AS InsPolNumber, COALESCE(VoidFlag, 0) AS void_flag,
         ROW_NUMBER() OVER (PARTITION BY TRIM(InsPolicyCode)
                            ORDER BY LastUpdatedDate DESC NULLS LAST, PolicyCode DESC NULLS LAST) AS rn
  FROM ods.policytravel) WHERE rn = 1
;

CREATE OR REPLACE TABLE stg.pr30_travel AS
SELECT TRIM(p.PolicyCode) AS PolicyCode, TRIM(p.ClientCode) AS ClientCode,
       COALESCE(sc.SCV_Customer_Key, sl.SCV_Customer_Key) AS SCV_Customer_Key,
       p.PolicyStatusDesc, CAST(p.PolicyInceptDate AS DATE) AS PolicyInceptDate, CAST(p.PolicyRenewDate AS DATE) AS PolicyRenewDate,
       CAST(p.PolicyCancelDate AS DATE) AS PolicyCancelDate,
       DATEDIFF(p.PolicyRenewDate, p.PolicyInceptDate) AS duration_days,
       p.CCYGrossPremium AS premium, p.CCYCommission AS commission, p.CCYFees AS fee,
       COALESCE(v.void_flag, 0) AS void_flag
FROM   ods.policy p
LEFT JOIN stg.pr30_scv_client  sc ON sc.ClientCode  = TRIM(p.ClientCode)
LEFT JOIN stg.pr30_scv_link    sl ON sl.PolicyCode  = TRIM(p.PolicyCode)
LEFT JOIN stg.pr30_travel_void v  ON v.InsPolNumber = TRIM(p.InsPolNumber)
WHERE  p.PolicyTypeGroup = 'Travel'
AND    p.PolicyInceptDate BETWEEN DATE'1990-01-01' AND DATE'2030-01-01'
;

CREATE OR REPLACE TABLE stg.pr30_travel_due AS
SELECT *,
       CASE WHEN PolicyCancelDate >= PolicyInceptDate AND PolicyCancelDate < PolicyRenewDate
            THEN PolicyCancelDate ELSE PolicyRenewDate END                                            AS end_date,
       CASE WHEN PolicyCancelDate IS NULL OR PolicyCancelDate >= PolicyRenewDate THEN 'reached renewal date'
            WHEN PolicyCancelDate >= date_sub(PolicyRenewDate, ${MatchDays})                         THEN 'cancelled near renewal'
            ELSE 'cancelled mid-term' END                                                            AS end_type,
       CASE WHEN PolicyCancelDate < date_sub(PolicyRenewDate, ${MatchDays}) THEN 0 ELSE 1 END           AS due_flag
FROM   stg.pr30_travel
WHERE  duration_days >= 45 AND premium > 0 AND PolicyRenewDate >= ${FirstDueDate}
;

Candidate pairs: term x new policy of the same customer and product whose renewal date is within ±MatchDays of (end_date + 12 months), i.e. it starts within ±MatchDays of end_date. end_date = the cancellation date if cancelled, else the renewal date. All terms are 12 months. tier: 1 same code renewed, 2 EDW lineage, 3 same vehicle / address, 4 date only. link_rank: 1 renewal (the term reached renewal or was cancelled within MatchDays of it), 2 rebook (cancelled mid-term and replaced). Renewals are matched first.

In [ ]:
CREATE OR REPLACE TABLE stg.pr30_match_pairs AS
SELECT d.product, d.SCV_Customer_Key, d.policy_code AS due_code, d.due_date, d.end_date,
       s.cand_code, concat(s.cand_code, '|', CAST(s.ReportingSaleDate AS STRING)) AS cand_key,
       s.term_start AS cand_start, s.ReportingSaleDate AS cand_sale_date, s.ReportingSaleType AS cand_sale_type,
       CASE WHEN s.cand_code = d.policy_code                                               THEN 1
            WHEN s.ly_code = d.policy_code OR s.cand_code = d.edw_ty_policy_code           THEN 2
            WHEN d.product IN ('Motor', 'Van') AND s.reg IS NOT NULL AND s.reg = d.reg     THEN 3
            WHEN d.product = 'Home' AND s.addr_key IS NOT NULL AND s.addr_key = d.addr_key THEN 3
            ELSE 4 END                                                             AS tier,
       abs(datediff(s.term_start, d.end_date))                                     AS gap_days,
       CAST(s.EURGrossPremium AS DOUBLE) AS cand_premium,
       CAST(s.EURCommission AS DOUBLE)   AS cand_commission,
       CAST(s.EURFees AS DOUBLE)         AS cand_fee,
       s.cand_due_date,
       CASE WHEN d.due_flag = 1 THEN 1 ELSE 2 END                                  AS link_rank
FROM   stg.pr30_due d
JOIN   stg.pr30_sale_cand s
       ON  s.SCV_Customer_Key = d.SCV_Customer_Key
       AND s.product = d.product
       AND s.term_start BETWEEN date_sub(d.end_date, ${MatchDays}) AND date_add(d.end_date, ${MatchDays})
       AND (s.cand_code = d.policy_code
            OR (NOT (s.ReportingSaleType = 'Renewals' AND s.ly_code IS NOT NULL AND s.ly_code <> d.policy_code)
                AND COALESCE(s.cand_status, '') NOT IN ('Temporary', 'NB Not Taken Up')))
UNION ALL
SELECT 'Travel', d.SCV_Customer_Key, d.PolicyCode, d.PolicyRenewDate, d.end_date,
       v.PolicyCode, v.PolicyCode, v.PolicyInceptDate, v.PolicyInceptDate, 'New Business',
       4, abs(datediff(v.PolicyInceptDate, d.end_date)),
       CAST(v.premium AS DOUBLE), CAST(v.commission AS DOUBLE), CAST(v.fee AS DOUBLE),
       v.PolicyRenewDate, CASE WHEN d.due_flag = 1 THEN 1 ELSE 2 END
FROM   stg.pr30_travel_due d
JOIN   stg.pr30_travel v
       ON  v.SCV_Customer_Key = d.SCV_Customer_Key
       AND v.PolicyCode <> d.PolicyCode
       AND v.PolicyStatusDesc IN ('New Business', 'Renewal')
       AND v.premium > 0 AND v.void_flag = 0
       AND v.PolicyInceptDate BETWEEN date_sub(d.end_date, ${MatchDays}) AND date_add(d.end_date, ${MatchDays})
;

In [ ]:
%%pyspark
# One-to-one matching per customer and product: renewals before rebooks, then the best tier, then the nearest start
# date; codes break ties so a rerun gives the same answer. Each term gets at most one new policy and each new policy
# is used at most once.
import pandas as pd

pairs = spark.table("stg.pr30_match_pairs")

def one_to_one(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.sort_values(["link_rank", "tier", "gap_days", "due_code", "cand_key"]).reset_index(drop=True)
    due_keys = pdf["due_code"] + "|" + pdf["due_date"].astype(str)
    used_due, used_cand, keep = set(), set(), []
    for i, (d, c) in enumerate(zip(due_keys, pdf["cand_key"])):
        if d in used_due or c in used_cand:
            continue
        used_due.add(d)
        used_cand.add(c)
        keep.append(i)
    return pdf.iloc[keep]

def per_partition(batches):
    # One Python call per partition (not per customer: ~470k groups made applyInPandas slow). The repartition puts each
    # customer and product in a single partition; the groups are then matched one by one, exactly as before.
    parts = list(batches)
    if not parts:
        return
    pdf = pd.concat(parts, ignore_index=True)
    out = [one_to_one(g) for _, g in pdf.groupby(["product", "SCV_Customer_Key"], sort=False, dropna=False)]
    if out:
        yield pd.concat(out, ignore_index=True)

(pairs.repartition(200, "product", "SCV_Customer_Key")
      .mapInPandas(per_partition, schema=pairs.schema)
      .write.mode("overwrite").format("delta").option("overwriteSchema", "true")
      .saveAsTable("stg.pr30_match"))

One row per policy term, with its link to the next term (chain columns are added in the next cell). link_type: 'renewal' (retained = 1) or 'rebook' (cancelled mid-term and replaced; rebooked = 1, not a renewal).

In [ ]:
CREATE OR REPLACE TABLE stg.pr30_out AS
WITH last_sale AS (SELECT MAX(ReportingSaleDate) AS d FROM edw.tbl_fact_policy_sales),
mhv AS (
    SELECT d.SCV_Customer_Key, d.product, d.ClientCode AS client_code,
           d.policy_code, d.term_start, d.due_date, d.cancel_date, d.end_date, d.end_type, d.due_flag,
           d.edw_offered, d.edw_retained, d.edw_ty_policy_code, d.edw_offered_premium, d.edw_offered_fee
    FROM   stg.pr30_due d
),
trv AS (
    SELECT t.SCV_Customer_Key, 'Travel' AS product, t.ClientCode AS client_code,
           t.PolicyCode AS policy_code, t.PolicyInceptDate AS term_start, t.PolicyRenewDate AS due_date,
           t.PolicyCancelDate AS cancel_date, t.end_date, t.end_type, t.due_flag,
           CAST(NULL AS INT) AS edw_offered, CAST(NULL AS INT) AS edw_retained, CAST(NULL AS STRING) AS edw_ty_policy_code,
           CAST(NULL AS DECIMAL(38, 6)) AS edw_offered_premium, CAST(NULL AS DECIMAL(38, 6)) AS edw_offered_fee
    FROM   stg.pr30_travel_due t
),
u AS (SELECT * FROM mhv UNION ALL SELECT * FROM trv)
SELECT u.SCV_Customer_Key, u.product, u.client_code, u.policy_code, u.term_start, u.due_date, u.cancel_date,
       u.end_date, u.end_type, u.due_flag,
       CASE WHEN m.cand_code IS NOT NULL AND u.due_flag = 1 THEN 1 ELSE 0 END AS retained,
       CASE WHEN m.cand_code IS NOT NULL AND u.due_flag = 0 THEN 1 ELSE 0 END AS rebooked,
       CASE WHEN m.cand_code IS NULL THEN NULL WHEN u.due_flag = 1 THEN 'renewal' ELSE 'rebook' END AS link_type,
       m.cand_code                                                            AS new_policy_code,
       m.cand_due_date                                                        AS new_due_date,
       CASE WHEN m.cand_code = u.policy_code THEN 1 WHEN m.cand_code IS NOT NULL THEN 0 END AS new_policy_same_code,
       m.cand_start                                                           AS new_term_start,
       m.cand_sale_date                                                       AS new_sale_date,
       m.cand_sale_type                                                       AS new_sale_type,
       m.tier                                                                 AS match_tier,
       CASE m.tier WHEN 1 THEN 'same code' WHEN 2 THEN 'EDW lineage' WHEN 3 THEN 'same vehicle / address'
                   WHEN 4 THEN 'date only' END                                AS match_evidence,
       m.gap_days                                                             AS match_gap_days,
       m.cand_premium AS new_premium, m.cand_commission AS new_commission, m.cand_fee AS new_fees,
       u.edw_offered, u.edw_retained, u.edw_ty_policy_code, u.edw_offered_premium, u.edw_offered_fee,
       CASE WHEN u.SCV_Customer_Key IS NULL                                     THEN 'no customer key'
            WHEN u.due_flag = 0                                                 THEN 'not due'
            WHEN m.cand_code IS NULL AND date_add(u.end_date, ${MatchDays}) > (SELECT d FROM last_sale) THEN 'pending'
            ELSE 'final' END                                                  AS outcome_status
FROM   u
LEFT JOIN stg.pr30_match m
       ON m.product = u.product AND m.due_code = u.policy_code AND m.due_date = u.due_date
;

In [ ]:
%%pyspark
# Policy chains: root policy -> next -> ... -> current, through renewal and rebook links (Vijay, 2026-10-13: the chain
# runs through a rebook; the rebook itself is still not a renewal). A link points at the next term by its policy code and renewal date; the term row is the one with that
# code whose due date is nearest, within 31 days. Per customer and product: chain_seq 1 = the root term.
import pandas as pd
from pyspark.sql import types as T

out = spark.table("stg.pr30_out")
extra = [T.StructField("previous_policy_code", T.StringType()), T.StructField("previous_due_date", T.DateType()),
         T.StructField("root_policy_code", T.StringType()), T.StructField("root_due_date", T.DateType()),
         T.StructField("chain_seq", T.IntegerType())]
schema = T.StructType(out.schema.fields + extra)

def chain(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.sort_values(["due_date", "policy_code"]).reset_index(drop=True)
    by_code = {}
    for i, (c, d) in enumerate(zip(pdf["policy_code"], pdf["due_date"])):
        by_code.setdefault(c, []).append((d, i))
    nxt, prev = {}, {}
    links = pdf[pdf["link_type"].isin(["renewal", "rebook"]) & pdf["new_policy_code"].notna() & pdf["new_due_date"].notna()]
    for i, c, d in zip(links.index, links["new_policy_code"], links["new_due_date"]):
        cands = [(abs((t - d).days), j) for t, j in by_code.get(c, []) if j != i and abs((t - d).days) <= 31]
        if not cands:
            continue
        j = min(cands)[1]
        if j in prev or pdf.at[j, "due_date"] <= pdf.at[i, "due_date"]:
            continue                       # one predecessor per term; links only run forward in time
        nxt[i], prev[j] = j, i
    n = len(pdf)
    root, seq = [None] * n, [None] * n
    for i in range(n):
        if i in prev:
            continue
        k, s = i, 1
        while True:
            root[k], seq[k] = i, s
            if k not in nxt:
                break
            k, s = nxt[k], s + 1
    pdf["previous_policy_code"] = [pdf.at[prev[i], "policy_code"] if i in prev else None for i in range(n)]
    pdf["previous_due_date"] = [pdf.at[prev[i], "due_date"] if i in prev else None for i in range(n)]
    pdf["root_policy_code"] = [pdf.at[r, "policy_code"] for r in root]
    pdf["root_due_date"] = [pdf.at[r, "due_date"] for r in root]
    pdf["chain_seq"] = pd.array(seq, dtype="Int32")
    return pdf

def per_partition(batches):
    # One Python call per partition, as in the matching step; groups (including rows with no customer key) as before.
    parts = list(batches)
    if not parts:
        return
    pdf = pd.concat(parts, ignore_index=True)
    out = [chain(g.copy()) for _, g in pdf.groupby(["product", "SCV_Customer_Key"], sort=False, dropna=False)]
    if out:
        yield pd.concat(out, ignore_index=True)[[f.name for f in schema.fields]]

(out.repartition(200, "product", "SCV_Customer_Key")
    .mapInPandas(per_partition, schema=schema)
    .write.mode("overwrite").format("delta").option("overwriteSchema", "true")
    .saveAsTable("stg.seg_policy_renewals_30d"))

## Checks

Grain and matching integrity (expect 0, 0, 0).

In [ ]:
SELECT (SELECT COUNT(*) - COUNT(DISTINCT product, policy_code, due_date) FROM stg.seg_policy_renewals_30d) AS duplicate_rows,
       (SELECT COUNT(*) - COUNT(DISTINCT product, due_code, due_date) FROM stg.pr30_match)                AS due_used_twice,
       (SELECT COUNT(*) - COUNT(DISTINCT product, SCV_Customer_Key, cand_key) FROM stg.pr30_match)      AS new_policy_used_twice;

By year and product: due, retained, against EDW's offered / retained flags.

In [ ]:
SELECT year(due_date) AS due_year, product, COUNT(*) AS terms, SUM(due_flag) AS due, SUM(retained) AS retained,
       SUM(1 - due_flag) AS cancelled_mid_term, SUM(rebooked) AS rebooked,
       ROUND(100.0 * SUM(retained) / NULLIF(SUM(due_flag), 0), 1) AS retention_pct,
       SUM(CASE WHEN edw_offered = 1 THEN 1 ELSE 0 END)  AS edw_offered,
       SUM(CASE WHEN edw_retained = 1 THEN 1 ELSE 0 END) AS edw_retained,
       SUM(CASE WHEN retained = 1 AND edw_retained = 1 THEN 1 ELSE 0 END) AS both_retained,
       SUM(CASE WHEN outcome_status = 'pending' THEN 1 ELSE 0 END)        AS pending,
       SUM(CASE WHEN outcome_status = 'no customer key' THEN 1 ELSE 0 END) AS no_customer_key
FROM   stg.seg_policy_renewals_30d
GROUP  BY year(due_date), product
ORDER  BY due_year, product;

Chains: every renewal / rebook link reaches the next term (expect linked = renewals + rebooks, or close).

In [ ]:
SELECT product,
       SUM(retained)                                                    AS renewals,
       SUM(rebooked)                                                    AS rebooks,
       SUM(CASE WHEN previous_policy_code IS NOT NULL THEN 1 ELSE 0 END) AS linked_from_previous,
       COUNT(DISTINCT root_policy_code, root_due_date)                  AS chains,
       MAX(chain_seq)                                                   AS longest_chain,
       SUM(CASE WHEN chain_seq > 1 AND root_policy_code <> policy_code THEN 1 ELSE 0 END) AS terms_on_a_changed_code
FROM   stg.seg_policy_renewals_30d
GROUP  BY product ORDER BY product;

How retained terms were matched, by product.

In [ ]:
SELECT product, match_evidence, new_policy_same_code, COUNT(*) AS terms, ROUND(AVG(match_gap_days), 1) AS avg_gap_days
FROM   stg.seg_policy_renewals_30d WHERE retained = 1
GROUP  BY product, match_evidence, new_policy_same_code ORDER BY product, match_evidence;